# Robust Journey Planner Speed Benchmark

Controls timing benchmarks for `RobustJourneyPlanner`.

**Workflow**
1. Run **Setup** once per kernel session.
2. Run **Prepare robust planner** to build/load CSA data and load the delay model.
3. After editing `src/routing/robust_journey_planner.py`, run the **reload code only** cell. It keeps prepared in-memory data and swaps in the latest planner code where possible.
4. Run any benchmark cell. Set `iterations` to reduce noise from fluctuations.

The instrumented source code prints per-function breakdowns. The bench functions add aggregate stats and compact route summaries.

## Setup

In [43]:
import importlib
import os
import sys

cwd = os.path.abspath(os.getcwd())
project_root = cwd if os.path.exists(os.path.join(cwd, "src")) else os.path.abspath(os.path.join(cwd, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import src.config.settings as _s
import src.data.csa_data_handler as _cdh
import src.models.delay_model as _dm
import src.models.delay_model_trainer as _dmt
import src.models.model_artifacts as _ma
import src.routing.robust_journey_planner as _rjp
import tests.test_robust_journey as _bench


def _prepared_summary(p):
    if p is None or not getattr(p, "prepared", False):
        return " (robust planner is not prepared yet)"

    days = getattr(p, "connections_by_day", None) or {}
    n_day_connections = sum(len(v) for v in days.values())
    n_stops = len(getattr(p, "stops", None) or [])
    n_trips = getattr(p, "n_trips", 0)
    has_delays = bool(getattr(p, "delay_lookup", None))
    return f" ({n_stops:,} stops, {n_trips:,} trips, {n_day_connections:,} day-connections, delay_lookup={has_delays})"


def reload_robust_code(preserve_prepared=True):
    """Reload robust planner/benchmark code while keeping prepared data in memory."""
    global settings, get_settings, RobustJourneyPlanner
    global bench_robust_prepare, bench_robust_plan
    global _s, _cdh, _dm, _dmt, _ma, _rjp, _bench, robust_planner

    old_planner = globals().get("robust_planner")

    for module in (_s, _cdh, _dm, _dmt, _ma, _rjp, _bench):
        importlib.reload(module)

    get_settings = _s.get_settings
    RobustJourneyPlanner = _rjp.RobustJourneyPlanner
    bench_robust_prepare = _bench.bench_robust_prepare
    bench_robust_plan = _bench.bench_robust_plan

    try:
        settings = get_settings()
    except Exception:
        if old_planner is not None and hasattr(old_planner, "settings"):
            settings = old_planner.settings
        elif "settings" in globals():
            settings = globals()["settings"]
        else:
            raise

    new_planner = RobustJourneyPlanner(settings=settings)

    if preserve_prepared and old_planner is not None and getattr(old_planner, "prepared", False):
        for name, value in vars(old_planner).items():
            setattr(new_planner, name, value)

        new_planner.settings = settings

        if getattr(new_planner, "data_handler", None) is not None:
            try:
                new_planner.data_handler.__class__ = _cdh.CSADataHandler
            except TypeError:
                pass

        robust_planner = new_planner
        print("Reloaded robust_journey_planner.py; preserved prepared data" + _prepared_summary(robust_planner))
    else:
        robust_planner = new_planner
        print("Reloaded robust_journey_planner.py; created a fresh unprepared robust planner")

    return robust_planner


robust_planner = reload_robust_code(preserve_prepared=True)

Reloaded robust_journey_planner.py; preserved prepared data (390 stops, 52,529 trips, 1,788,536 day-connections, delay_lookup=False)


In [44]:
# Run this after editing src/routing/robust_journey_planner.py or helpers.
# This swaps in latest code while keeping prepared data in memory when possible.
robust_planner = reload_robust_code()

Reloaded robust_journey_planner.py; preserved prepared data (390 stops, 52,529 trips, 1,788,536 day-connections, delay_lookup=False)


In [45]:
LAUSANNE_REGION_UUIDS = (
    "a7a21b73-6ffe-4fbf-a635-6e2b961f3072",
    "e168fd57-f57a-4075-a350-0dcfbb55147f",
)
REGION_UUIDS = settings.region_uuids or LAUSANNE_REGION_UUIDS

START_STOP = 8501120   # Lausanne
END_STOP   = 8501117   # Renens VD
TRAVEL_DATE = "2026-05-27"
DEADLINE = "18:00"
CONFIDENCE_Q = 0.90
iterations = 6

print(f"Regions    : {REGION_UUIDS}")
print(f"Stops      : {START_STOP} -> {END_STOP}")
print(f"Date       : {TRAVEL_DATE}")
print(f"Deadline   : {DEADLINE}")
print(f"Confidence : {CONFIDENCE_Q}")
print(f"Iterations : {iterations}")

Regions    : ('a7a21b73-6ffe-4fbf-a635-6e2b961f3072', 'e168fd57-f57a-4075-a350-0dcfbb55147f')
Stops      : 8501120 -> 8501117
Date       : 2026-05-27
Deadline   : 18:00
Confidence : 0.9
Iterations : 6


## Prepare robust planner

Run this only when `robust_planner.prepared` is `False`, or when you intentionally want to rebuild/refetch data.

- `FORCE_REBUILD = True` recreates Trino tables.
- `REBUILD_CSA_PREREQUISITES = True` rebuilds stops/footpaths prerequisites.
- `TRAIN_DELAY_MODEL = True` trains the delay model and is normally slow.
- After editing only Python code, run the reload-code cell above instead of this cell.

In [46]:
FORCE_PREPARE = True
FORCE_REBUILD = True
REBUILD_CSA_PREREQUISITES = True
TRAIN_DELAY_MODEL = False
FORCE_RETRAIN_DELAY_MODEL = False

if robust_planner.prepared and not FORCE_PREPARE:
    print("robust_planner already prepared; skipping prepare(). Set FORCE_PREPARE=True to run it again.")
    robust_prepare_result = {"skipped": True}
else:
    robust_planner = reload_robust_code(preserve_prepared=False)
    robust_prepare_result = bench_robust_prepare(
        robust_planner,
        regions=REGION_UUIDS,
        force_rebuild=FORCE_REBUILD,
        rebuild_csa_prerequisites=REBUILD_CSA_PREREQUISITES,
        train_delay_model=TRAIN_DELAY_MODEL,
        force_retrain_delay_model=FORCE_RETRAIN_DELAY_MODEL,
        travel_date="2026-05-27"
    )

robust_prepare_result

Reloaded robust_journey_planner.py; created a fresh unprepared robust planner

  bench_robust_prepare
  LOAD DELAY LOOKUP
  Loaded 2/2 district files
  delay_lookup.load              10.99s  (803706 keys)

  delay model loaded  (7 quantile boosters)

  BUILD CSA TABLES
  creating  stops...
  build_stops                     0.89s
  creating  stop_to_stop...
  build_footpaths                 0.81s
  creating  stop_times_seq...
  build_stop_times_seq            3.51s
  creating  stop_times_trips_seq...
  build_stop_times_trips_seq      3.86s
  creating  full_table_seq...
  build_full_table_seq            3.88s
  creating  connections...
  build_connections               4.60s
--------------------------------------------
  total build_all                17.55s

  FETCH DATA
  fetch_stops                     0.08s  (390 rows)
  fetch_footpaths                 0.10s  (1492 rows)
  fetch_connections              12.01s  (620440 rows)

  MATERIALIZE IN-MEMORY
  stops dict                      

{'total_sec': 53.798572395928204}

In [47]:
planner = robust_planner
print(f"robust_planner.prepared = {robust_planner.prepared}")
print(f"delay_lookup loaded     = {bool(robust_planner.delay_lookup)}")
print(f"n_trips                 = {planner.n_trips}")
print(f"n_stops                 = {len(planner.stops)}")

robust_planner.prepared = True
delay_lookup loaded     = True
n_trips                 = 52529
n_stops                 = 390


## Benchmark single route

Full robust plan: scheduled candidates, delay prediction, confidence evaluation, final route sorting.

In [48]:
robust_plan_result = bench_robust_plan(
    robust_planner,
    iterations=iterations,
    start_stop_id=START_STOP,
    end_stop_id=END_STOP,
    travel_date=TRAVEL_DATE,
    arrival_deadline=DEADLINE,
    confidence_q=CONFIDENCE_Q,
    
    max_routes=5,
)
robust_plan_result


  bench_robust_plan  (n=6)
  run  1/6    plan                total=163.7ms  scanned=32260  routes=5
  164.89 ms   routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000
  run  2/6      0.03 ms   routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000
  run  3/6      0.02 ms   routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000
  run  4/6      0.02 ms   routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000
  run  5/6      0.02 ms   routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000
  run  6/6      0.02 ms   routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000
--------------------------------------------
  avg                        27.50 ms
  min                         0.02 ms
  max                       164.89 ms
  stdev                      67.31 ms
--------------------------------------------
  bench_robust_plan done     27.50 ms



{'times_ms': [164.88912398926914,
  0.030410941690206528,
  0.02025999128818512,
  0.02098199911415577,
  0.020737992599606514,
  0.022363848984241486],
 'avg_ms': 27.500646460490923,
 'min_ms': 0.02025999128818512,
 'max_ms': 164.88912398926914,
 'stdev_ms': 67.30633340365836,
 'n_routes': [5, 5, 5, 5, 5, 5],
 'route_summaries': ['routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000',
  'routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000',
  'routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000',
  'routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000',
  'routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000',
  'routes=5 best=17:38:00->17:48:00 robust_arr=17:50:41 conf=1.000']}

## Benchmark random busy-stop pairs

In [49]:
import random
import statistics
from collections import Counter

random.seed(42)

stop_freq = Counter()
for conn in planner.connections_by_day["monday"]:
    stop_freq[conn[0]] += 1
    stop_freq[conn[1]] += 1

busy_stops = [s for s, _ in stop_freq.most_common(200)]

TEST_PAIRS = []
while len(TEST_PAIRS) < 100:
    s, e = random.choice(busy_stops), random.choice(busy_stops)
    if s != e:
        TEST_PAIRS.append((s, e))

len(TEST_PAIRS), TEST_PAIRS[:5]

(100,
 [(8504171, 8592102),
  (8579237, 8579235),
  (8592071, 8501120),
  (8592209, 8592125),
  (8587853, 8592017)])

In [40]:
# Run this after editing src/routing/robust_journey_planner.py or helpers.
# This swaps in latest code while keeping prepared data in memory when possible.
robust_planner = reload_robust_code()

Reloaded robust_journey_planner.py; preserved prepared data (390 stops, 52,529 trips, 1,786,782 day-connections, delay_lookup=False)


In [52]:
all_times = []
all_routes = []

for i, (start_stop, end_stop) in enumerate(TEST_PAIRS):
    print(f"PAIR {i + 1}/{len(TEST_PAIRS)}: {start_stop} -> {end_stop}")
    result = bench_robust_plan(
        robust_planner,
        iterations=1,
        start_stop_id=start_stop,
        end_stop_id=end_stop,
        travel_date="2026-05-27",
        arrival_deadline=DEADLINE,
        confidence_q=0.1,
        max_routes=5,
        search_window_minutes=120
    )
    all_times.append(result["avg_ms"])
    all_routes.append(result["n_routes"][-1] if result["n_routes"] else 0)

all_times.sort()
print(f"  n:    {len(all_times)}")
print(f"  avg:  {statistics.mean(all_times):.1f}ms")
print(f"  p50:  {all_times[len(all_times)//2]:.1f}ms")
print(f"  p95:  {all_times[int(len(all_times)*0.95)]:.1f}ms")
print(f"  max:  {all_times[-1]:.1f}ms")
print(f"  avg routes returned: {statistics.mean(all_routes):.2f}")

PAIR 1/100: 8504171 -> 8592102

  bench_robust_plan  (n=1)
  run  1/1    plan                total=160.0ms  scanned=25586  routes=5
  161.36 ms   routes=5 best=17:13:00->17:50:00 robust_arr=17:53:01 conf=0.453
--------------------------------------------
  avg                       161.36 ms
  min                       161.36 ms
  max                       161.36 ms
  stdev                       0.00 ms
--------------------------------------------
  bench_robust_plan done    161.36 ms

PAIR 2/100: 8579237 -> 8579235

  bench_robust_plan  (n=1)
  run  1/1    plan                total=57.3ms  scanned=18703  routes=5
   57.92 ms   routes=5 best=17:33:00->17:56:00 robust_arr=17:59:39 conf=0.310
--------------------------------------------
  avg                        57.92 ms
  min                        57.92 ms
  max                        57.92 ms
  stdev                       0.00 ms
--------------------------------------------
  bench_robust_plan done     57.92 ms

PAIR 3/100: 8592071

## Summary table


 - n:    30
 - avg:  325.7ms
 - p50:  346.2ms
 - p95:  574.7ms
- max:  798.4ms
- avg routes returned: 0.90



The journey planner:

  - n:    30
  - avg:  260.7ms
  - p50:  282.5ms
  - p95:  318.5ms
  - max:  320.3ms
  - avg routes returned: 0.33

The confidence optimized journey_planner:
  - n:    30
  - avg:  579.8ms
  - p50:  607.8ms
  - p95:  814.4ms
  - max:  854.7ms
  - avg routes returned: 1.70



Lausanne only:

 -  n:    30
 -  avg:  78.8ms
 -  p50:  79.2ms
 -  p95:  86.9ms
 -  max:  104.1ms
 -  avg routes returned: 5.00

1 route run
  - n:    100
  - avg:  306.9ms
  - p50:  304.2ms
  - p95:  341.6ms
  - max:  387.6ms


With early stopping

  - n:    100
  - avg:  87.6ms
  - p50:  74.7ms
  - p95:  270.7ms
  - max:  340.8ms
  - avg routes returned: 4.91


With on connection confidences.
  - n:    100
  - avg:  67.9ms
  - p50:  56.0ms
  - p95:  188.8ms
  - max:  226.6ms
  - avg routes returned: 4.97

Final with some changes, (0.75 confidence):
  - n:    100
  - avg:  86.1ms
  - p50:  95.8ms
  - p95:  123.0ms
  - max:  161.7ms
  - avg routes returned: 4.19 | with confidence 0.1 -> 4.93 routes

In [53]:
import pandas as pd

rows = []
for label, r in [
    ("robust_plan", robust_plan_result),
]:
    rows.append({
        "benchmark": label,
        "avg_ms": round(r["avg_ms"], 2),
        "min_ms": round(r["min_ms"], 2),
        "max_ms": round(r["max_ms"], 2),
        "stdev_ms": round(r["stdev_ms"], 2),
        "routes": r["n_routes"],
    })

pd.DataFrame(rows).set_index("benchmark")

,avg_ms,min_ms,max_ms,stdev_ms,routes
benchmark,,,,,
robust_plan,27.5,0.02,164.89,67.31,"[5, 5, 5, 5, 5, 5]"
